# Proyecto final · 01 · Bronze

**Columna vertebral, parte 1 · Sección 2 del examen.** Ingesta **incremental** desde el volume `landing` hacia una tabla Delta gobernada.

Lo que se evalúa:
- La ingesta usa Auto Loader (o `COPY INTO`) y guarda el esquema en `schemaLocation`.
- Una segunda corrida **solo carga los archivos nuevos**. La evidencia es la celda de conteo por lote, corrida después de cargar dos lotes.
- Cada fila sabe de qué archivo vino.

In [0]:
# TU CÓDIGO ACÁ
origen = "/Volumes/pf_fallas/bronze/landing/vehiculos/"
tabla_bronze = "pf_fallas.bronze.vehiculos_raw"

schema_location = "/Volumes/pf_fallas/bronze/landing/_schemas/vehiculos_raw"
checkpoint = "/Volumes/pf_fallas/bronze/landing/_checkpoints/vehiculos_raw"

In [0]:
from pyspark.sql import functions as F

(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("pathGlobFilter", "*.csv")
    .load(origen)
    .withColumn("archivo_origen", F.col("_metadata.file_path"))
    .withColumn("fecha_ingesta", F.current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(tabla_bronze)
    .awaitTermination()
)

### Evidencia de la ingesta incremental
Corre esta celda después de cargar dos lotes. Cada lote debe aparecer **una sola vez**, con su propia hora de ingesta.

In [0]:
# Consultar los registros y fechas de ingesta por lote.
# Esta consulta no carga archivos ni modifica los datos.
display(spark.sql("""
    SELECT
        regexp_extract(archivo_origen, 'lote_[0-9]+', 0) AS lote,
        COUNT(*) AS filas,
        MIN(fecha_ingesta) AS primera_ingesta,
        MAX(fecha_ingesta) AS ultima_ingesta
    FROM pf_fallas.bronze.vehiculos_raw
    GROUP BY 1
    ORDER BY 1
"""))

In [0]:

%sql
SELECT
    COUNT(*) AS filas_totales,
    COUNT(DISTINCT vehicle_id) AS vehiculos_unicos,
    COUNT(*) - COUNT(DISTINCT vehicle_id) AS registros_repetidos
FROM pf_fallas.bronze.vehiculos_raw;

### TU RESPUESTA
¿Qué pasaría si corres esta ingesta dos veces seguidas sin archivos nuevos? ¿Qué guarda el checkpoint para que eso pase?

Si la ingesta se ejecuta nuevamente sin archivos nuevos, no se agregan filas
y el conteo permanece igual.

El `checkpoint` conserva el progreso de la ingesta y, junto con el estado
de Auto Loader, permite reconocer los archivos ya procesados.

Al incorporar el segundo lote, solamente se cargaron sus `15.010` filas.
Bronze conserva los `10` vehículos repetidos entre archivos; estos se
eliminarán en Silver usando `vehicle_id` como llave de negocio.